# 04 - Image Retrieval Debug

Notebook này dùng để **kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản**. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.

### BƯỚC 1: Setup Tối Thiểu

- **Tác dụng chính:** Notebook này dùng để kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `find_chatbot_fashion_root`, `CHATBOT_FASHION_DIR` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


> **Ranh giới kiến trúc:** notebook này không định nghĩa top-level router. Các helper tại đây chỉ suy ra constraint/retrieval goal bên trong một pipeline đã được `app/core/intent.py` chọn. Vì vậy chúng không làm tăng số business intent hay execution route.


In [34]:
import json

import math

import random

import re

import sys

import unicodedata

import uuid

from pathlib import Path

import numpy as np

from langchain_core.documents import Document

from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient

from qdrant_client.http.models import Distance, FieldCondition, Filter, MatchValue, VectorParams

from qdrant_client.models import PointStruct

from tqdm.auto import tqdm

def find_chatbot_fashion_root(start: Path | None = None) -> Path:
    """Xử lý bước `find chatbot fashion root` của pipeline.

    Args:
        start (Path | None): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        Path: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "app" / "config.py").exists():
            return candidate
    raise RuntimeError("Không tìm thấy thư mục gốc Chatbot_Fashion chứa app/config.py")


In [35]:
CHATBOT_FASHION_DIR = find_chatbot_fashion_root()

if str(CHATBOT_FASHION_DIR) not in sys.path:
    sys.path.insert(0, str(CHATBOT_FASHION_DIR))

from app.config import (
    IMAGE_EMBEDDING_BATCH_SIZE,
    IMAGE_SEARCH_MAX_PRODUCTS as APP_IMAGE_SEARCH_MAX_PRODUCTS,
    IMAGE_SEARCH_SCORE_THRESHOLD as APP_IMAGE_SEARCH_SCORE_THRESHOLD,
    IMAGE_SEARCH_TOP_K as APP_IMAGE_SEARCH_TOP_K,
    IMAGE_VECTOR_SIZE,
    METADATA_FILE as APP_METADATA_FILE,
    OLLAMA_BASE_URL,
    PRODUCT_IMAGE_ROOT as APP_PRODUCT_IMAGE_ROOT,
    PRODUCT_SEARCH_BRAND_LIMIT as APP_PRODUCT_SEARCH_BRAND_LIMIT,
    PRODUCT_SEARCH_CANDIDATE_K as APP_PRODUCT_SEARCH_CANDIDATE_K,
    PRODUCT_SEARCH_PAGE_SIZE as APP_PRODUCT_SEARCH_PAGE_SIZE,
    QWEN_VL_MODEL,
    QDRANT_COLLECTION_FASHION,
    QDRANT_COLLECTION_PRODUCT_IMAGE,
    QDRANT_URL,
)

print("[OK] Setup notebook 04 hoàn tất")

print(f"Project root: {CHATBOT_FASHION_DIR}")


[OK] Setup notebook 04 hoàn tất
Project root: D:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion


### BƯỚC 2: Cấu Hình Image Retrieval

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `PRODUCT_COLLECTION`, `PRODUCT_IMAGE_COLLECTION`, `IMAGE_SEARCH_TOP_K`, `IMAGE_SEARCH_MAX_PRODUCTS`, `IMAGE_SEARCH_SCORE_THRESHOLD` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [36]:
PRODUCT_COLLECTION = QDRANT_COLLECTION_FASHION

PRODUCT_IMAGE_COLLECTION = QDRANT_COLLECTION_PRODUCT_IMAGE

IMAGE_SEARCH_TOP_K = APP_IMAGE_SEARCH_TOP_K

IMAGE_SEARCH_MAX_PRODUCTS = APP_IMAGE_SEARCH_MAX_PRODUCTS

IMAGE_SEARCH_SCORE_THRESHOLD = APP_IMAGE_SEARCH_SCORE_THRESHOLD

TEXT_SEARCH_CANDIDATE_K = APP_PRODUCT_SEARCH_CANDIDATE_K

TEXT_SEARCH_MAX_PRODUCTS = APP_PRODUCT_SEARCH_PAGE_SIZE

TEXT_SEARCH_BRAND_LIMIT = APP_PRODUCT_SEARCH_BRAND_LIMIT

MULTIMODAL_FINAL_K = IMAGE_SEARCH_MAX_PRODUCTS

CATEGORY_LOCK_MIN_SHARE = 0.50

CATEGORY_LOCK_MIN_COUNT = 2

CONFIDENCE_SCORE_GAP_MIN = 0.02

VLM_RERANK_TOP_K = 5

VLM_OLLAMA_BASE_URL = OLLAMA_BASE_URL

_image_embeddings_instance = None

_text_embeddings_instance = None

_text_vector_db = None

def get_image_embeddings():
    """Load FashionCLIP image encoder đúng lúc cần encode ảnh, không load ở đầu notebook.

    Args:
        Không có.

    Returns:
        Any: Vector hoặc tensor biểu diễn dữ liệu đầu vào.
    """
    global _image_embeddings_instance
    if _image_embeddings_instance is None:
        from app.core.image_search import FashionCLIPImageEmbeddings

        print("[INFO] Load FashionCLIP image encoder lần đầu...")
        _image_embeddings_instance = FashionCLIPImageEmbeddings(batch_size=IMAGE_EMBEDDING_BATCH_SIZE)
    return _image_embeddings_instance

def get_text_vector_db():
    """Load text embedding/vector store khi cần so sánh image retrieval với text retrieval.

    Args:
        Không có.

    Returns:
        Any: Kết quả đã xử lý của hàm.
    """
    global _text_vector_db
    if _text_vector_db is None:
        print("[INFO] Chuẩn bị text vector store để so sánh với image retrieval...")
        _text_vector_db = QdrantVectorStore(
            client=qdrant,
            collection_name=PRODUCT_COLLECTION,
            embedding=get_text_embeddings(),
        )
    return _text_vector_db

def get_text_embeddings():
    """Load ViFashionCLIP text embedder 512-dim, cùng không gian với FashionCLIP image vector.

    Args:
        Không có.

    Returns:
        Any: Vector hoặc tensor biểu diễn dữ liệu đầu vào.
    """
    global _text_embeddings_instance
    if _text_embeddings_instance is None:
        from app.core.embeddings import get_product_embeddings

        _text_embeddings_instance = get_product_embeddings()
    return _text_embeddings_instance


In [37]:
PRODUCT_IMAGE_ROOT = Path(APP_PRODUCT_IMAGE_ROOT)

METADATA_FILE = Path(APP_METADATA_FILE)

MULTIMODAL_CANDIDATE_K = max(IMAGE_SEARCH_TOP_K, 30)

qdrant = QdrantClient(url=QDRANT_URL, timeout=20, check_compatibility=False)

print("[OK] Image retrieval config")

print(f"Image root : {PRODUCT_IMAGE_ROOT}")

print(f"Metadata   : {METADATA_FILE}")

print(f"Collection : {PRODUCT_IMAGE_COLLECTION}")

print(f"Search     : top-{IMAGE_SEARCH_TOP_K} | threshold={IMAGE_SEARCH_SCORE_THRESHOLD} | final={IMAGE_SEARCH_MAX_PRODUCTS}")

print(f"Text compare: top-{TEXT_SEARCH_CANDIDATE_K} -> final {TEXT_SEARCH_MAX_PRODUCTS}")

print(f"Multimodal : candidates={MULTIMODAL_CANDIDATE_K} -> final {MULTIMODAL_FINAL_K} | VLM top-{VLM_RERANK_TOP_K}")

print(f"VLM model  : {QWEN_VL_MODEL}")

print(f"VLM Ollama : {VLM_OLLAMA_BASE_URL}")


[OK] Image retrieval config
Image root : D:\KHÓA LUẬN\WORKSPACE\Amazon_Lazada_Fashion_Metadata_65k\images
Metadata   : D:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion\data\metadata\meta_Amazon_Lazada_Fashion_65k.jsonl
Collection : fashion_products_fashionclip_image_main_65k
Search     : top-12 | threshold=0.15 | final=3
Text compare: top-15 -> final 5
Multimodal : candidates=30 -> final 3 | VLM top-5
VLM model  : qwen2.5vl:3b
VLM Ollama : http://localhost:11434


### BƯỚC 3: Tìm Ảnh MAIN Của Sản Phẩm

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `get_main_image_relative_path`, `resolve_main_image_path`, `iter_products_with_main_image` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [38]:
def get_main_image_relative_path(item: dict) -> str:
    """Chọn ảnh MAIN theo thứ tự ưu tiên: variant MAIN, tên có _MAIN, rồi ảnh đầu tiên.

    Args:
        item (dict): Bản ghi sản phẩm hoặc đối tượng đang xử lý.

    Returns:
        str: Kết quả đã xử lý của hàm.
    """
    images = item.get("images", []) or []

    for image in images:
        if str(image.get("variant", "")).upper() == "MAIN" and image.get("large"):
            return image["large"]

    for image in images:
        large = str(image.get("large", ""))
        if "_MAIN" in large.upper():
            return large

    if images and images[0].get("large"):
        return images[0]["large"]

    product_id = item.get("product_id")
    return f"images/{product_id}_MAIN.jpg" if product_id else ""

def resolve_main_image_path(relative_path: str, image_root: str | Path = PRODUCT_IMAGE_ROOT) -> Path:
    """Bỏ prefix `images/` nếu có, rồi ghép với thư mục ảnh local.

    Args:
        relative_path (str): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        image_root (str | Path): Ảnh hoặc thông tin ảnh đầu vào.

    Returns:
        Path: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    rel = Path(str(relative_path).replace("\\", "/"))
    if rel.parts and rel.parts[0].lower() == "images":
        rel = Path(*rel.parts[1:])
    return Path(image_root) / rel

def iter_products_with_main_image(
    metadata_file: str | Path = METADATA_FILE,
    image_root: str | Path = PRODUCT_IMAGE_ROOT,
):
    """Đọc JSONL và yield `(item, rel_path, img_path)` cho sản phẩm có ảnh MAIN tồn tại.

    Args:
        metadata_file (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        image_root (str | Path): Ảnh hoặc thông tin ảnh đầu vào.

    Returns:
        Any: Kết quả đã xử lý của hàm.
    """
    metadata_file = Path(metadata_file)
    image_root = Path(image_root)
    with metadata_file.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue

            rel_path = get_main_image_relative_path(item)
            img_path = resolve_main_image_path(rel_path, image_root)
            if img_path.exists():
                yield item, rel_path, img_path


In [39]:
print("[OK] Image path helpers ready")


[OK] Image path helpers ready


### BƯỚC 4: Preview Một Ảnh MAIN Bất Kỳ

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `preview_random_main_image`, `sample` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [40]:
def preview_random_main_image(
    metadata_file: str | Path = METADATA_FILE,
    image_root: str | Path = PRODUCT_IMAGE_ROOT,
):
    """Random một sản phẩm có ảnh MAIN tồn tại để kiểm tra path trước khi index/search.

    Args:
        metadata_file (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        image_root (str | Path): Ảnh hoặc thông tin ảnh đầu vào.

    Returns:
        Any: Kết quả đã xử lý của hàm.
    """
    chosen = None
    count = 0
    for item, rel_path, img_path in iter_products_with_main_image(metadata_file, image_root):
        count += 1
        if random.randrange(count) == 0:
            chosen = (item, rel_path, img_path)

    if chosen is None:
        print("[WARN] Không tìm thấy sản phẩm nào có ảnh MAIN tồn tại.")
        return None

    item, rel_path, img_path = chosen
    print(f"Scanned products with MAIN image: {count}")
    print(f"product_id : {item.get('product_id', '')}")
    print(f"title      : {item.get('title', '')}")
    print(f"category   : {item.get('category', '')}")
    print(f"rel_path   : {rel_path}")
    print(f"img_path   : {img_path}")
    print(f"exists     : {img_path.exists()}")
    return {"item": item, "rel_path": rel_path, "img_path": img_path}


# Ví dụ:


In [41]:
sample = preview_random_main_image()


Scanned products with MAIN image: 65480
product_id : B072PQ1NFB
title      : Khuyên tai dáng dài YAZILIND đính đá Zirconia
category   : Bông tai
rel_path   : images/B072PQ1NFB_MAIN.jpg
img_path   : D:\KHÓA LUẬN\WORKSPACE\Amazon_Lazada_Fashion_Metadata_65k\images\B072PQ1NFB_MAIN.jpg
exists     : True


### BƯỚC 5: Build Payload Ảnh Cho Qdrant

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `extract_image_urls`, `normalize_to_text`, `build_product_metadata`, `build_product_page_content`, `build_image_payload` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [42]:
def extract_image_urls(item: dict) -> list[str]:
    """Lấy các ảnh chất lượng cao từ metadata sản phẩm."""
    return [image.get("large") for image in item.get("images", []) if image.get("large")]

def normalize_to_text(value, default: str = "Không rõ") -> str:
    """Đổi giá trị metadata thành text ổn định để đưa vào page_content.

    Args:
        value (Any): Giá trị đầu vào phục vụ bước xử lý này.
        default (str): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        str: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    if value is None or value == "":
        return default
    if isinstance(value, list):
        cleaned = [str(x).strip() for x in value if str(x).strip()]
        return ", ".join(cleaned) if cleaned else default
    return str(value).strip() or default

def build_product_metadata(item: dict) -> dict:
    """Tạo metadata gọn, ổn định cho một sản phẩm.

    Args:
        item (dict): Bản ghi sản phẩm hoặc đối tượng đang xử lý.

    Returns:
        dict: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    image_urls = extract_image_urls(item)
    return {
        "product_id": item.get("product_id", ""),
        "title": item.get("title", ""),
        "category": item.get("category", ""),
        "department": item.get("department", ""),
        "brand": item.get("brand", ""),
        "price": item.get("price", 0),
        "images": image_urls,
        "image_url": image_urls[0] if image_urls else "",
    }

def build_product_page_content(item: dict) -> str:
    """Tạo text mô tả sản phẩm để LLM có context sau khi image search trả về kết quả.

    Args:
        item (dict): Bản ghi sản phẩm hoặc đối tượng đang xử lý.

    Returns:
        str: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    details = item.get("details", {}) or {}
    fields = [
        ("Tên sản phẩm", item.get("title")),
        ("Mã sản phẩm", item.get("product_id")),
        ("Danh mục", item.get("category")),
        ("Đối tượng", item.get("department")),
        ("Thương hiệu", item.get("brand")),
        ("Giá", f"{item.get('price', 0)} VND"),
        ("Màu sắc", details.get("main_color")),
        ("Chất liệu", details.get("material")),
        ("Kích cỡ", details.get("size")),
        ("Dịp sử dụng", item.get("occasion")),
        ("Mô tả", item.get("description")),
    ]
    return "\n".join(f"{label}: {normalize_to_text(value)}" for label, value in fields)

def build_image_payload(item: dict, rel_path: str, img_path: Path) -> dict:
    """Tạo payload lưu vào Qdrant cùng vector ảnh MAIN.

    Args:
        item (dict): Bản ghi sản phẩm hoặc đối tượng đang xử lý.
        rel_path (str): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        img_path (Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.

    Returns:
        dict: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    metadata = build_product_metadata(item)
    metadata["main_image_path"] = str(img_path)
    metadata["main_image_relpath"] = rel_path
    metadata["image_url"] = metadata.get("image_url") or rel_path
    return {
        "product_id": metadata.get("product_id", ""),
        "title": metadata.get("title", ""),
        "category": metadata.get("category", ""),
        "department": metadata.get("department", ""),
        "brand": metadata.get("brand", ""),
        "price": metadata.get("price", 0),
        "image_url": metadata.get("image_url", ""),
        "main_image_path": str(img_path),
        "main_image_relpath": rel_path,
        "page_content": build_product_page_content(item),
        "metadata": metadata,
    }

def preview_image_payload(sample: dict | None) -> dict | None:
    """In payload rút gọn để kiểm tra trước khi index vào Qdrant.

    Args:
        sample (dict | None): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        dict | None: Kết quả đã xử lý của hàm.
    """
    if not sample:
        return None
    payload = build_image_payload(sample["item"], sample["rel_path"], sample["img_path"])
    print("product_id:", payload["product_id"])
    print("title     :", payload["title"])
    print("category  :", payload["category"])
    print("image_url :", payload["image_url"])
    print("content   :", payload["page_content"][:300].replace("\n", " | "))
    return payload


# Ví dụ:


In [43]:
payload = preview_image_payload(sample)


product_id: B072PQ1NFB
title     : Khuyên tai dáng dài YAZILIND đính đá Zirconia
category  : Bông tai
image_url : images/B072PQ1NFB_MAIN.jpg
content   : Tên sản phẩm: Khuyên tai dáng dài YAZILIND đính đá Zirconia | Mã sản phẩm: B072PQ1NFB | Danh mục: Bông tai | Đối tượng: Nữ | Thương hiệu: YAZILIND | Giá: 95760 VND | Màu sắc: Vàng | Chất liệu: Mạ vàng 18K, đá Cubic Zirconia | Kích cỡ: Kích cỡ chung | Dịp sử dụng: Xã hội, Hàng ngày | Mô tả: Đôi khuyên tai dáng dài sang 


### BƯỚC 6: Kiểm Tra Image Collection Trong Qdrant

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `check_image_collection`, `image_collection_info` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [44]:
def check_image_collection(collection_name: str = PRODUCT_IMAGE_COLLECTION) -> dict | None:
    """Kiểm tra collection ảnh có tồn tại và đang có bao nhiêu point.

    Args:
        collection_name (str): Tên collection Qdrant đích.

    Returns:
        dict | None: Kết quả kiểm tra hoặc báo cáo chẩn đoán.
    """
    if not qdrant.collection_exists(collection_name):
        print(f"[WARN] Chưa có image collection: {collection_name}")
        return None

    count = qdrant.count(collection_name).count
    info = qdrant.get_collection(collection_name)
    print(f"[OK] Collection: {collection_name}")
    print(f"Points     : {count}")
    print(f"Status     : {getattr(info, 'status', '')}")
    return {"collection": collection_name, "points": count, "info": info}


# Ví dụ:


In [45]:
image_collection_info = check_image_collection()


[OK] Collection: fashion_products_fashionclip_image_main_65k
Points     : 65330
Status     : green


### BƯỚC 7: Index Ảnh MAIN Vào Qdrant

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `run_main_image_index_pipeline` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [46]:
def run_main_image_index_pipeline(
    metadata_file: str | Path = METADATA_FILE,
    image_root: str | Path = PRODUCT_IMAGE_ROOT,
    collection_name: str = PRODUCT_IMAGE_COLLECTION,
    batch_size: int = 64,
) -> None:
    """Index một ảnh MAIN cho mỗi sản phẩm vào Qdrant, có resume theo số point hiện có.

    Args:
        metadata_file (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        image_root (str | Path): Ảnh hoặc thông tin ảnh đầu vào.
        collection_name (str): Tên collection Qdrant đích.
        batch_size (int): Giới hạn số phần tử được xử lý hoặc trả về.

    Returns:
        None: Kết quả đã xử lý của hàm.
    """
    metadata_file = Path(metadata_file)
    image_root = Path(image_root)
    if not metadata_file.exists():
        raise FileNotFoundError(f"Không tìm thấy metadata: {metadata_file}")
    if not image_root.exists():
        raise FileNotFoundError(f"Không tìm thấy thư mục ảnh: {image_root}")

    items = list(iter_products_with_main_image(metadata_file, image_root))
    print(f"[OK] Tìm thấy {len(items)} sản phẩm có ảnh MAIN")

    if not qdrant.collection_exists(collection_name):
        qdrant.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=IMAGE_VECTOR_SIZE, distance=Distance.COSINE),
        )
        current_count = 0
    else:
        current_count = qdrant.count(collection_name).count
        print(f"[INFO] Collection đã có {current_count} vectors")

    remaining = items[current_count:]
    if not remaining:
        print("[OK] Image collection đã được index đầy đủ.")
        return

    embeddings = get_image_embeddings()
    with tqdm(total=len(items), initial=current_count, desc="Image index", unit="img") as progress:
        # Chia dữ liệu thành các batch nhỏ để giới hạn bộ nhớ khi index.
        for start in range(0, len(remaining), batch_size):
            batch = remaining[start:start + batch_size]
            image_paths = [img_path for _, _, img_path in batch]
            vectors = embeddings.encode_image_paths(image_paths)
            points = []

            for (item, rel_path, img_path), vector in zip(batch, vectors):
                if vector is None:
                    continue
                product_id = str(item.get("product_id", ""))
                point_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f"fashion-main-image:{product_id}"))
                points.append(PointStruct(
                    id=point_id,
                    vector=vector,
                    payload=build_image_payload(item, rel_path, img_path),
                ))

            if points:
                qdrant.upsert(collection_name=collection_name, points=points)
            progress.update(len(batch))

    print(f"[OK] Index ảnh hoàn tất -> {collection_name}")


# Chỉ bỏ comment khi thật sự muốn index lại:
# run_main_image_index_pipeline()


### BƯỚC 8: Search Bằng Ảnh, Text Và Vector Ghép

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `normalize_product_metadata`, `image_point_to_document`, `query_image_collection_by_vector`, `search_products_by_image`, `diversity_filter_documents` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [47]:
def normalize_product_metadata(doc: Document) -> Document:
    """Đảm bảo Document có `images` dạng list và `image_url` ổn định.

    Args:
        doc (Document): Document hoặc danh sách Document cần xử lý.

    Returns:
        Document: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    images = doc.metadata.get("images", [])
    if isinstance(images, str):
        images = [images] if images else []
    doc.metadata["images"] = images
    doc.metadata["image_url"] = doc.metadata.get("image_url") or (images[0] if images else "")
    return doc


def image_point_to_document(point, score_key: str = "image_search_score") -> Document:
    """Đổi Qdrant image point thành Document dùng chung với prompt LLM.

    Args:
        point (Any): Giá trị đầu vào phục vụ bước xử lý này.
        score_key (str): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.

    Returns:
        Document: Kết quả đã xử lý của hàm.
    """
    payload = point.payload or {}
    metadata = payload.get("metadata", {}) or {}
    metadata[score_key] = getattr(point, "score", None)
    metadata["image_url"] = payload.get("image_url") or metadata.get("image_url", "")
    return normalize_product_metadata(Document(
        page_content=payload.get("page_content", ""),
        metadata=metadata,
    ))


def query_image_collection_by_vector(
    query_vector: list[float],
    collection_name: str = PRODUCT_IMAGE_COLLECTION,
    top_k: int = IMAGE_SEARCH_TOP_K,
    max_products: int = IMAGE_SEARCH_MAX_PRODUCTS,
    score_threshold: float | None = IMAGE_SEARCH_SCORE_THRESHOLD,
    score_key: str = "image_search_score",
    query_filter: Filter | None = None,
) -> list[Document]:
    """Query image collection bằng một vector 512-dim đã normalize.

    Args:
        query_vector (list[float]): Nội dung truy vấn hoặc văn bản đầu vào.
        collection_name (str): Tên collection Qdrant đích.
        top_k (int): Giới hạn số phần tử được xử lý hoặc trả về.
        max_products (int): Bản ghi sản phẩm hoặc đối tượng đang xử lý.
        score_threshold (float | None): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.
        score_key (str): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.
        query_filter (Filter | None): Nội dung truy vấn hoặc văn bản đầu vào.

    Returns:
        list[Document]: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    if not qdrant.collection_exists(collection_name):
        print(f"[WARN] Collection ảnh chưa tồn tại: {collection_name}")
        return []

    kwargs = {
        "collection_name": collection_name,
        "query": query_vector,
        "limit": top_k,
        "with_payload": True,
    }
    if score_threshold is not None:
        kwargs["score_threshold"] = score_threshold
    if query_filter is not None:
        kwargs["query_filter"] = query_filter

    # Truy vấn Qdrant để lấy candidate cùng score phục vụ debug.
    response = qdrant.query_points(**kwargs)
    docs: list[Document] = []
    seen_ids: set[str] = set()
    for point in response.points:
        doc = image_point_to_document(point, score_key=score_key)
        product_id = str(doc.metadata.get("product_id", ""))
        if product_id and product_id in seen_ids:
            continue
        docs.append(doc)
        if product_id:
            seen_ids.add(product_id)
        if len(docs) >= max_products:
            break
    return docs


def search_products_by_image(
    image_path: str | Path,
    collection_name: str = PRODUCT_IMAGE_COLLECTION,
    top_k: int = IMAGE_SEARCH_TOP_K,
    max_products: int = IMAGE_SEARCH_MAX_PRODUCTS,
    score_threshold: float | None = IMAGE_SEARCH_SCORE_THRESHOLD,
) -> list[Document]:
    """Tìm sản phẩm tương tự ảnh query, lọc trùng product_id và giới hạn số kết quả cuối.

    Args:
        image_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        collection_name (str): Tên collection Qdrant đích.
        top_k (int): Giới hạn số phần tử được xử lý hoặc trả về.
        max_products (int): Bản ghi sản phẩm hoặc đối tượng đang xử lý.
        score_threshold (float | None): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.

    Returns:
        list[Document]: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    image_path = Path(image_path)
    if not image_path.exists():
        print(f"[LỖI] Không tìm thấy ảnh: {image_path}")
        return []
    if not qdrant.collection_exists(collection_name):
        print(f"[WARN] Collection ảnh chưa tồn tại: {collection_name}")
        return []

    query_vector = get_image_embeddings().embed_image(image_path)
    if query_vector is None:
        print("[WARN] Không encode được ảnh query.")
        return []

    return query_image_collection_by_vector(
        query_vector=query_vector,
        collection_name=collection_name,
        top_k=top_k,
        max_products=max_products,
        score_threshold=score_threshold,
        score_key="image_search_score",
    )


def diversity_filter_documents(
    docs: list[Document],
    max_docs: int = TEXT_SEARCH_MAX_PRODUCTS,
    max_per_brand: int = TEXT_SEARCH_BRAND_LIMIT,
) -> list[Document]:
    """Lọc trùng product_id và hạn chế quá nhiều sản phẩm cùng brand.

    Args:
        docs (list[Document]): Document hoặc danh sách Document cần xử lý.
        max_docs (int): Document hoặc danh sách Document cần xử lý.
        max_per_brand (int): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        list[Document]: Kết quả đã xử lý của hàm.
    """
    selected: list[Document] = []
    seen_product_ids: set[str] = set()
    brand_counts: dict[str, int] = {}

    for doc in docs:
        doc = normalize_product_metadata(doc)
        product_id = str(doc.metadata.get("product_id", "")).strip().lower()
        brand = str(doc.metadata.get("brand", "")).strip().lower()

        if product_id and product_id in seen_product_ids:
            continue
        if brand and brand_counts.get(brand, 0) >= max_per_brand:
            continue

        selected.append(doc)
        if product_id:
            seen_product_ids.add(product_id)
        if brand:
            brand_counts[brand] = brand_counts.get(brand, 0) + 1
        if len(selected) >= max_docs:
            return selected

    return selected


def search_products_by_text(
    text_query: str,
    k: int = TEXT_SEARCH_CANDIDATE_K,
    max_products: int = TEXT_SEARCH_MAX_PRODUCTS,
) -> list[Document]:
    """Chạy text retrieval gọn để đối chiếu với image retrieval trong cùng notebook.

    Args:
        text_query (str): Nội dung truy vấn hoặc văn bản đầu vào.
        k (int): Giá trị đầu vào phục vụ bước xử lý này.
        max_products (int): Bản ghi sản phẩm hoặc đối tượng đang xử lý.

    Returns:
        list[Document]: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    if not text_query or not text_query.strip():
        return []

    vector_db = get_text_vector_db()
    pairs = vector_db.similarity_search_with_score(query=text_query, k=k)
    docs: list[Document] = []
    for rank, (doc, score) in enumerate(pairs, start=1):
        doc = normalize_product_metadata(doc)
        doc.metadata["text_rank"] = rank
        doc.metadata["text_score"] = float(score)
        docs.append(doc)
    return diversity_filter_documents(docs, max_docs=max_products)


def strip_vietnamese_accents(text: str) -> str:
    """Đưa text về dạng không dấu để bắt keyword ổn hơn khi user gõ thiếu dấu."""
    normalized = unicodedata.normalize("NFD", str(text))
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn").replace("đ", "d").replace("Đ", "D")


def normalize_debug_text(text: str) -> str:
    """Lowercase + bỏ dấu + gom khoảng trắng, chỉ dùng cho rule debug đơn giản."""
    text = strip_vietnamese_accents(text).lower()
    return re.sub(r"\s+", " ", text).strip()


def infer_retrieval_goal(text_query: str) -> str:
    """Suy ra retrieval goal cục bộ: search/similar giữ loại sản phẩm, outfit thì không khóa cứng category.

    Args:
        text_query (str): Nội dung truy vấn hoặc văn bản đầu vào.

    Returns:
        str: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    q = normalize_debug_text(text_query)
    outfit_keywords = [
        "phoi", "outfit", "mix", "mac voi", "di voi", "ket hop", "goi y set", "hop voi",
        "nen mac", "mac cung", "style voi",
    ]
    if any(keyword in q for keyword in outfit_keywords):
        return "outfit"
    return "search"


def infer_target_departments(text_query: str) -> list[str]:
    """Bắt tín hiệu giới tính/departement từ câu hỏi để boost mềm, không dùng làm filter cứng.

    Args:
        text_query (str): Nội dung truy vấn hoặc văn bản đầu vào.

    Returns:
        list[str]: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    q = normalize_debug_text(text_query)
    if any(keyword in q for keyword in ["nu", "female", "woman", "women", "girl", "feminine", "nang", "ban gai"]):
        return ["Nữ", "Unisex"]
    if any(keyword in q for keyword in ["nam", "male", "man", "men", "boy", "ban trai"]):
        return ["Nam", "Unisex"]
    return []


def infer_category_from_docs(
    docs: list[Document],
    min_share: float = CATEGORY_LOCK_MIN_SHARE,
    min_count: int = CATEGORY_LOCK_MIN_COUNT,
) -> dict:
    """Lấy category chiếm đa số trong image-only results để quyết định có nên khóa category không.

    Args:
        docs (list[Document]): Document hoặc danh sách Document cần xử lý.
        min_share (float): Giá trị đầu vào phục vụ bước xử lý này.
        min_count (int): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        dict: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    counts: dict[str, int] = {}
    for doc in docs:
        category = str(doc.metadata.get("category", "")).strip()
        if category:
            counts[category] = counts.get(category, 0) + 1

    total = sum(counts.values())
    if not counts or total == 0:
        return {"category": None, "share": 0.0, "count": 0, "counts": counts, "confident": False}

    category, count = max(counts.items(), key=lambda item: item[1])
    share = count / total
    confident = count >= min_count and share >= min_share
    return {"category": category, "share": share, "count": count, "counts": counts, "confident": confident}


def build_category_filter(category: str | None) -> Filter | None:
    """Tạo Qdrant filter cho payload top-level `category` trong image collection.

    Args:
        category (str | None): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.

    Returns:
        Filter | None: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    if not category:
        return None
    # Lọc payload Qdrant theo ràng buộc đã suy ra trước khi xếp hạng.
    return Filter(must=[FieldCondition(key="category", match=MatchValue(value=category))])


def score_gap(docs: list[Document], score_key: str) -> float | None:
    """Khoảng cách score giữa rank 1 và rank 2; gap nhỏ nghĩa là top result chưa thật sự nổi bật.

    Args:
        docs (list[Document]): Document hoặc danh sách Document cần xử lý.
        score_key (str): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.

    Returns:
        float | None: Kết quả đã xử lý của hàm.
    """
    scores = [doc.metadata.get(score_key) for doc in docs if doc.metadata.get(score_key) is not None]
    if len(scores) < 2:
        return None
    return float(scores[0]) - float(scores[1])


def metadata_rerank_documents(
    docs: list[Document],
    text_query: str,
    locked_category: str | None = None,
    category_mode: str = "none",
    intent: str = "search",
    final_k: int = MULTIMODAL_FINAL_K,
) -> list[Document]:
    """Rerank nhẹ bằng metadata: category từ ảnh, department từ text, và score retrieval gốc.

    Args:
        docs (list[Document]): Document hoặc danh sách Document cần xử lý.
        text_query (str): Nội dung truy vấn hoặc văn bản đầu vào.
        locked_category (str | None): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        category_mode (str): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        intent (str): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        final_k (int): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        list[Document]: Kết quả đã xử lý của hàm.
    """
    target_departments = infer_target_departments(text_query)
    reranked: list[Document] = []

    for doc in docs:
        doc = normalize_product_metadata(doc)
        meta = doc.metadata
        base_score = (
            meta.get("composed_score")
            if meta.get("composed_score") is not None
            else meta.get("image_search_score")
            if meta.get("image_search_score") is not None
            else meta.get("text_score")
            if meta.get("text_score") is not None
            else 0.0
        )
        try:
            score = float(base_score)
        except (TypeError, ValueError):
            score = 0.0

        reasons = [f"base={score:.3f}"]
        category = str(meta.get("category", "")).strip()
        department = str(meta.get("department", "") or meta.get("gender", "")).strip()

        if locked_category:
            if category == locked_category:
                bonus = 0.20 if category_mode == "hard_lock" else 0.12
                score += bonus
                reasons.append(f"category={locked_category} +{bonus:.2f}")
            elif category_mode == "hard_lock":
                score -= 0.50
                reasons.append(f"khác category {locked_category} -0.50")

        if target_departments:
            if department in target_departments:
                score += 0.10
                reasons.append(f"department={department} +0.10")
            elif not department and any(word in normalize_debug_text(doc.page_content) for word in [" nu ", " nu,", " nu.", "female", "women"]):
                score += 0.05
                reasons.append("text có tín hiệu nữ +0.05")

        if intent == "outfit" and locked_category and category != locked_category:
            score += 0.05
            reasons.append("outfit cần mở category +0.05")

        meta["metadata_rerank_score"] = score
        meta["metadata_rerank_reason"] = "; ".join(reasons)
        reranked.append(doc)

    reranked.sort(key=lambda doc: float(doc.metadata.get("metadata_rerank_score", 0.0)), reverse=True)
    return reranked[:final_k]


def l2_normalize_vector(vector: list[float] | np.ndarray) -> list[float]:
    """Normalize vector về độ dài 1 để cosine search ổn định.

    Args:
        vector (list[float] | np.ndarray): Client hoặc vector store dùng để truy vấn.

    Returns:
        list[float]: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    arr = np.asarray(vector, dtype=np.float32)
    # Chuẩn hóa vector về cùng thang đo để phép so sánh cosine ổn định.
    norm = float(np.linalg.norm(arr))
    if not math.isfinite(norm) or norm <= 0:
        raise ValueError("Vector rỗng hoặc norm bằng 0, không thể normalize.")
    return (arr / norm).astype(float).tolist()


def embed_image_query(image_path: str | Path) -> list[float]:
    """Encode ảnh query thành vector FashionCLIP 512-dim đã normalize.

    Args:
        image_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.

    Returns:
        list[float]: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    vector = get_image_embeddings().embed_image(image_path)
    if vector is None:
        raise RuntimeError(f"Không encode được ảnh: {image_path}")
    return l2_normalize_vector(vector)


def embed_text_query_for_image_space(text_query: str) -> list[float]:
    """Encode text query tiếng Việt thành vector 512-dim cùng không gian với image collection.

    Args:
        text_query (str): Nội dung truy vấn hoặc văn bản đầu vào.

    Returns:
        list[float]: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    if not text_query or not text_query.strip():
        raise ValueError("text_query đang rỗng.")
    vector = get_text_embeddings().embed_query(text_query)
    return l2_normalize_vector(vector)


def compose_image_text_vector(
    image_vector: list[float],
    text_vector: list[float],
    image_weight: float = 0.6,
    text_weight: float = 0.4,
) -> list[float]:
    """Ghép ảnh + text ở vector level: normalize(w_img * image + w_text * text).

    Args:
        image_vector (list[float]): Ảnh hoặc thông tin ảnh đầu vào.
        text_vector (list[float]): Nội dung truy vấn hoặc văn bản đầu vào.
        image_weight (float): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.
        text_weight (float): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.

    Returns:
        list[float]: Kết quả đã xử lý của hàm.
    """
    if image_weight < 0 or text_weight < 0:
        raise ValueError("image_weight và text_weight phải >= 0.")
    if image_weight == 0 and text_weight == 0:
        raise ValueError("Ít nhất một weight phải > 0.")

    image_arr = np.asarray(image_vector, dtype=np.float32)
    text_arr = np.asarray(text_vector, dtype=np.float32)
    if image_arr.shape != text_arr.shape:
        raise ValueError(f"Vector ảnh và text khác chiều: {image_arr.shape} vs {text_arr.shape}")

    # Trộn embedding ảnh và văn bản theo trọng số của truy vấn đa phương thức.
    composed = image_weight * image_arr + text_weight * text_arr
    return l2_normalize_vector(composed)


def search_products_by_composed_image_text(
    image_path: str | Path,
    text_query: str,
    image_weight: float = 0.6,
    text_weight: float = 0.4,
    top_k: int = IMAGE_SEARCH_TOP_K,
    max_products: int = IMAGE_SEARCH_MAX_PRODUCTS,
    score_threshold: float | None = None,
    query_filter: Filter | None = None,
) -> list[Document]:
    """True zero-shot composed retrieval: ảnh + text -> một vector chung -> image collection.

    Args:
        image_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        text_query (str): Nội dung truy vấn hoặc văn bản đầu vào.
        image_weight (float): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.
        text_weight (float): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.
        top_k (int): Giới hạn số phần tử được xử lý hoặc trả về.
        max_products (int): Bản ghi sản phẩm hoặc đối tượng đang xử lý.
        score_threshold (float | None): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.
        query_filter (Filter | None): Nội dung truy vấn hoặc văn bản đầu vào.

    Returns:
        list[Document]: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    image_vector = embed_image_query(image_path)
    text_vector = embed_text_query_for_image_space(text_query)
    composed_vector = compose_image_text_vector(
        image_vector=image_vector,
        text_vector=text_vector,
        image_weight=image_weight,
        text_weight=text_weight,
    )
    docs = query_image_collection_by_vector(
        query_vector=composed_vector,
        collection_name=PRODUCT_IMAGE_COLLECTION,
        top_k=top_k,
        max_products=max_products,
        score_threshold=score_threshold,
        score_key="composed_score",
        query_filter=query_filter,
    )
    for doc in docs:
        doc.metadata["image_weight"] = image_weight
        doc.metadata["text_weight"] = text_weight
    return docs


### BƯỚC 9: Debug Composed Retrieval Và Calibrate Threshold

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `image_doc_row`, `resolve_doc_image_path`, `show_image_file`, `show_retrieval_gallery`, `print_docs_table` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [48]:
def image_doc_row(doc: Document, rank: int) -> dict:
    """Rút gọn Document thành dict dễ đọc khi debug retrieval.

    Args:
        doc (Document): Document hoặc danh sách Document cần xử lý.
        rank (int): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        dict: Kết quả đã xử lý của hàm.
    """
    doc = normalize_product_metadata(doc)
    return {
        "rank": rank,
        "product_id": doc.metadata.get("product_id", ""),
        "title": doc.metadata.get("title", ""),
        "category": doc.metadata.get("category", ""),
        "brand": doc.metadata.get("brand", ""),
        "image_score": doc.metadata.get("image_search_score"),
        "text_score": doc.metadata.get("text_score"),
        "composed_score": doc.metadata.get("composed_score"),
        "metadata_rerank_score": doc.metadata.get("metadata_rerank_score"),
        "metadata_rerank_reason": doc.metadata.get("metadata_rerank_reason", ""),
        "vlm_score": doc.metadata.get("vlm_score"),
        "vlm_reason": doc.metadata.get("vlm_reason", ""),
        "image_url": doc.metadata.get("image_url", ""),
        "preview": doc.page_content[:220].replace("\n", " | "),
    }


def resolve_doc_image_path(doc: Document) -> Path | None:
    """Tìm file ảnh local tốt nhất từ metadata của Document.

    Args:
        doc (Document): Document hoặc danh sách Document cần xử lý.

    Returns:
        Path | None: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    doc = normalize_product_metadata(doc)

    main_image_path = doc.metadata.get("main_image_path")
    if main_image_path and Path(main_image_path).exists():
        return Path(main_image_path)

    image_url = doc.metadata.get("image_url", "")
    if image_url and not str(image_url).startswith(("http://", "https://")):
        candidate = resolve_main_image_path(str(image_url), PRODUCT_IMAGE_ROOT)
        if candidate.exists():
            return candidate

    images = doc.metadata.get("images", []) or []
    for image in images:
        if image and not str(image).startswith(("http://", "https://")):
            candidate = resolve_main_image_path(str(image), PRODUCT_IMAGE_ROOT)
            if candidate.exists():
                return candidate
    return None


def show_image_file(image_path: str | Path, title: str = "Ảnh query") -> None:
    """Hiển thị một ảnh local ngay trong notebook.

    Args:
        image_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        title (str): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        None: Không trả về dữ liệu mới hoặc trả trạng thái sau khi hiển thị debug.
    """
    from IPython.display import display
    from PIL import Image as PILImage

    image_path = Path(image_path)
    if not image_path.exists():
        print(f"[WARN] Không thấy ảnh để hiển thị: {image_path}")
        return
    print(title)
    display(PILImage.open(image_path).convert("RGB"))


def show_retrieval_gallery(title: str, docs: list[Document], score_key: str = "image_search_score") -> None:
    """Hiển thị ảnh top results thành gallery nhỏ để đánh giá retrieval bằng mắt.

    Args:
        title (str): Giá trị đầu vào phục vụ bước xử lý này.
        docs (list[Document]): Document hoặc danh sách Document cần xử lý.
        score_key (str): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.

    Returns:
        None: Không trả về dữ liệu mới hoặc trả trạng thái sau khi hiển thị debug.
    """
    if not docs:
        print(f"\n=== {title}: không có kết quả ===")
        return

    import matplotlib.pyplot as plt
    from PIL import Image as PILImage

    items = []
    for rank, doc in enumerate(docs, start=1):
        img_path = resolve_doc_image_path(doc)
        if img_path is None:
            continue
        items.append((rank, doc, img_path))

    if not items:
        print(f"[WARN] Có {len(docs)} kết quả nhưng không resolve được file ảnh local để hiển thị.")
        return

    cols = min(3, len(items))
    rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4.5 * rows))
    if rows == 1 and cols == 1:
        axes = [axes]
    elif rows == 1:
        axes = list(axes)
    else:
        axes = list(axes.ravel())

    for ax in axes:
        ax.axis("off")

    for ax, (rank, doc, img_path) in zip(axes, items):
        meta = doc.metadata
        score = meta.get(score_key)
        score_text = "" if score is None else f" | {float(score):.3f}"
        title_text = f"#{rank}{score_text}\n{meta.get('category', '')} | {meta.get('brand', '')}\n{meta.get('product_id', '')}"
        ax.imshow(PILImage.open(img_path).convert("RGB"))
        ax.set_title(title_text, fontsize=9)
        ax.axis("off")

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


def print_docs_table(title: str, docs: list[Document], score_key: str) -> None:
    """Xử lý bước `print docs table` của pipeline.

    Args:
        title (str): Giá trị đầu vào phục vụ bước xử lý này.
        docs (list[Document]): Document hoặc danh sách Document cần xử lý.
        score_key (str): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.

    Returns:
        None: Không trả về dữ liệu mới hoặc trả trạng thái sau khi hiển thị debug.
    """
    print(f"\n=== {title} ({len(docs)}) ===")
    for rank, doc in enumerate(docs, start=1):
        row = image_doc_row(doc, rank)
        score = doc.metadata.get(score_key)
        score_text = "" if score is None else f" score={float(score):.4f}"
        print(f"#{rank}{score_text} | {row['product_id']} | {row['category']} | {row['brand']}")
        print(f"  {row['title']}")
        if row["metadata_rerank_reason"]:
            print(f"  metadata_rerank: {row['metadata_rerank_reason']}")
        if row["vlm_reason"]:
            print(f"  vlm: score={row['vlm_score']} | {row['vlm_reason']}")
        print(f"  image_url: {row['image_url']}")
        print(f"  {row['preview']}")


def debug_image_query(
    image_path: str | Path,
    top_k: int = IMAGE_SEARCH_TOP_K,
    max_products: int = IMAGE_SEARCH_MAX_PRODUCTS,
    score_threshold: float | None = IMAGE_SEARCH_SCORE_THRESHOLD,
    show_images: bool = True,
) -> list[Document]:
    """Xử lý bước `debug image query` của pipeline.

    Args:
        image_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        top_k (int): Giới hạn số phần tử được xử lý hoặc trả về.
        max_products (int): Bản ghi sản phẩm hoặc đối tượng đang xử lý.
        score_threshold (float | None): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.
        show_images (bool): Cờ bật hoặc tắt hành vi tương ứng.

    Returns:
        list[Document]: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    print(f"IMAGE: {Path(image_path)}")
    if show_images:
        show_image_file(image_path, title="Ảnh query")

    docs = search_products_by_image(
        image_path=image_path,
        top_k=top_k,
        max_products=max_products,
        score_threshold=score_threshold,
    )
    print_docs_table("Image search results", docs, score_key="image_search_score")
    if show_images:
        show_retrieval_gallery("Top image retrieval results", docs, score_key="image_search_score")
    return docs


def compare_retrieval_sets(image_docs: list[Document], text_docs: list[Document]) -> dict:
    """So sánh nhanh overlap giữa kết quả image retrieval và text retrieval.

    Args:
        image_docs (list[Document]): Ảnh hoặc thông tin ảnh đầu vào.
        text_docs (list[Document]): Nội dung truy vấn hoặc văn bản đầu vào.

    Returns:
        dict: Kết quả đã xử lý của hàm.
    """
    image_ids = {str(doc.metadata.get("product_id", "")) for doc in image_docs if doc.metadata.get("product_id")}
    text_ids = {str(doc.metadata.get("product_id", "")) for doc in text_docs if doc.metadata.get("product_id")}
    image_categories = {str(doc.metadata.get("category", "")) for doc in image_docs if doc.metadata.get("category")}
    text_categories = {str(doc.metadata.get("category", "")) for doc in text_docs if doc.metadata.get("category")}

    overlap_ids = image_ids & text_ids
    overlap_categories = image_categories & text_categories
    summary = {
        "image_count": len(image_docs),
        "text_count": len(text_docs),
        "overlap_product_ids": sorted(overlap_ids),
        "overlap_categories": sorted(overlap_categories),
    }
    print("\n=== Image vs Text Retrieval Summary ===")
    print(f"Image docs        : {summary['image_count']}")
    print(f"Text docs         : {summary['text_count']}")
    print(f"Trùng product_id  : {summary['overlap_product_ids'] or 'Không có'}")
    print(f"Trùng category    : {summary['overlap_categories'] or 'Không có'}")
    return summary


### BƯỚC 9B: VLM Rerank Candidate

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `extract_json_object`, `clamp_score`, `get_vlm_ollama_client`, `vlm_score_candidate`, `vlm_rerank_documents` để các bước sau tiếp tục sử dụng.


In [49]:
def extract_json_object(text: str) -> dict:
    """Parse JSON từ VLM output, chịu được trường hợp model lỡ thêm vài chữ bên ngoài.

    Args:
        text (str): Nội dung truy vấn hoặc văn bản đầu vào.

    Returns:
        dict: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    text = str(text or "").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start : end + 1])
    raise ValueError(f"Không parse được JSON từ VLM output: {text[:200]}")


def clamp_score(value, default: float = 0.0) -> float:
    """Đưa score về khoảng 0..1 để VLM rerank không làm vỡ pipeline.

    Args:
        value (Any): Giá trị đầu vào phục vụ bước xử lý này.
        default (float): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        float: Kết quả đã xử lý của hàm.
    """
    try:
        score = float(value)
    except (TypeError, ValueError):
        score = default
    return max(0.0, min(1.0, score))


def get_vlm_ollama_client():
    """Tạo Ollama client cho VLM. Đổi `VLM_OLLAMA_BASE_URL` để trỏ sang Vast.ai nếu cần."""
    import ollama

    return ollama.Client(host=VLM_OLLAMA_BASE_URL)


def vlm_score_candidate(
    query_image_path: str | Path,
    text_query: str,
    candidate_doc: Document,
    intent: str = "search",
    locked_category: str | None = None,
) -> dict:
    """Dùng VLM chấm một candidate bằng ảnh query + ảnh candidate + câu user.

    Args:
        query_image_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        text_query (str): Nội dung truy vấn hoặc văn bản đầu vào.
        candidate_doc (Document): Document hoặc danh sách Document cần xử lý.
        intent (str): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        locked_category (str | None): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.

    Returns:
        dict: Kết quả đã xử lý của hàm.
    """
    candidate_image_path = resolve_doc_image_path(candidate_doc)
    if candidate_image_path is None:
        return {"score": 0.0, "reason": "Không resolve được ảnh candidate local."}

    from app.core.vision import _preprocess_image

    query_b64 = _preprocess_image(str(query_image_path))
    candidate_b64 = _preprocess_image(str(candidate_image_path))
    meta = candidate_doc.metadata
    prompt = f"""
Bạn là reranker sản phẩm thời trang.
Ảnh 1 là ảnh query của người dùng. Ảnh 2 là sản phẩm candidate.

Yêu cầu người dùng: {text_query}
Intent đã suy ra: {intent}
Category khóa/boost từ ảnh query: {locked_category or "không có"}

Candidate metadata:
- title: {meta.get("title", "")}
- category: {meta.get("category", "")}
- brand: {meta.get("brand", "")}
- department: {meta.get("department", "") or meta.get("gender", "")}

Hãy chấm mức phù hợp của candidate với ảnh query và yêu cầu người dùng.
Nếu intent là search/similar item, ưu tiên giữ đúng loại sản phẩm từ ảnh query.
Nếu intent là outfit/phối đồ, candidate có thể khác loại sản phẩm nếu hợp để phối.
Trả về JSON duy nhất theo format:
{{"score": 0.0, "reason": "lý do ngắn dưới 25 từ"}}
""".strip()

    client = get_vlm_ollama_client()
    response = client.chat(
        model=QWEN_VL_MODEL,
        messages=[{"role": "user", "content": prompt, "images": [query_b64, candidate_b64]}],
        options={"temperature": 0},
    )
    content = response.get("message", {}).get("content", "")
    parsed = extract_json_object(content)
    return {
        "score": clamp_score(parsed.get("score")),
        "reason": str(parsed.get("reason", "")).strip()[:240],
    }


def vlm_rerank_documents(
    query_image_path: str | Path,
    text_query: str,
    docs: list[Document],
    intent: str = "search",
    locked_category: str | None = None,
    top_k: int = VLM_RERANK_TOP_K,
) -> list[Document]:
    """VLM rerank top candidates. Nếu VLM lỗi, giữ nguyên thứ tự metadata rerank để notebook vẫn chạy.

    Args:
        query_image_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        text_query (str): Nội dung truy vấn hoặc văn bản đầu vào.
        docs (list[Document]): Document hoặc danh sách Document cần xử lý.
        intent (str): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        locked_category (str | None): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        top_k (int): Giới hạn số phần tử được xử lý hoặc trả về.

    Returns:
        list[Document]: Kết quả đã xử lý của hàm.
    """
    reranked: list[Document] = []
    for rank, doc in enumerate(docs[:top_k], start=1):
        try:
            result = vlm_score_candidate(
                query_image_path=query_image_path,
                text_query=text_query,
                candidate_doc=doc,
                intent=intent,
                locked_category=locked_category,
            )
        except Exception as exc:
            result = {"score": 0.0, "reason": f"VLM lỗi ở rank {rank}: {type(exc).__name__}: {exc}"}
        doc.metadata["vlm_score"] = result["score"]
        doc.metadata["vlm_reason"] = result["reason"]
        reranked.append(doc)

    reranked.sort(key=lambda doc: float(doc.metadata.get("vlm_score", 0.0)), reverse=True)
    return reranked + docs[top_k:]


### BƯỚC 9C: Quyết Định Có Nên Hỏi Lại Người Dùng Không

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `build_clarification_diagnostic` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [50]:
def build_clarification_diagnostic(
    text_query: str,
    intent: str,
    category_mode: str,
    category_info: dict,
    composed_docs: list[Document],
    final_docs: list[Document],
) -> dict:
    """Search trước, rồi quyết định có nên hỏi lại user không.

    Args:
        text_query (str): Nội dung truy vấn hoặc văn bản đầu vào.
        intent (str): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        category_mode (str): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        category_info (dict): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        composed_docs (list[Document]): Document hoặc danh sách Document cần xử lý.
        final_docs (list[Document]): Document hoặc danh sách Document cần xử lý.

    Returns:
        dict: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    questions: list[str] = []
    composed_gap = score_gap(composed_docs, "composed_score")
    final_gap = score_gap(final_docs, "metadata_rerank_score")

    if not final_docs:
        questions.append("Bạn có thể mô tả rõ hơn sản phẩm muốn tìm không?")
    if intent == "search" and not category_info.get("confident"):
        questions.append("Ảnh này bạn muốn tìm đúng loại sản phẩm nào?")
    if composed_gap is not None and composed_gap < CONFIDENCE_SCORE_GAP_MIN:
        questions.append("Các kết quả đang khá sát nhau. Bạn ưu tiên giống kiểu dáng, màu sắc hay thương hiệu?")
    if infer_target_departments(text_query) and category_mode == "hard_lock":
        questions.append("Bạn muốn giữ đúng loại sản phẩm trong ảnh nhưng đổi sang phong cách/department nào?")

    diagnostic = {
        "needs_clarification": bool(questions),
        "questions": questions,
        "composed_score_gap": composed_gap,
        "final_score_gap": final_gap,
    }

    print("\n=== Clarification diagnostic ===")
    print(f"needs_clarification: {diagnostic['needs_clarification']}")
    print(f"composed_score_gap : {composed_gap}")
    print(f"final_score_gap    : {final_gap}")
    for question in questions[:3]:
        print(f"- {question}")
    return diagnostic


### BƯỚC 9D: Pipeline Chính Cho Ảnh + Text

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `debug_image_and_text`, `debug_multimodal_retrieval` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [51]:
def debug_image_and_text(
    image_path: str | Path,
    text_query: str,
    image_weight: float = 0.6,
    text_weight: float = 0.4,
    composed_threshold: float | None = None,
    intent: str = "auto",
    category_mode: str = "auto",
    use_vlm_rerank: bool = False,
    vlm_top_k: int = VLM_RERANK_TOP_K,
    show_images: bool = True,
) -> dict:
    """Debug 3 nhánh: image-only, text-only và true composed image+text retrieval.

    Args:
        image_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        text_query (str): Nội dung truy vấn hoặc văn bản đầu vào.
        image_weight (float): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.
        text_weight (float): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.
        composed_threshold (float | None): Giá trị điểm, ngưỡng hoặc trọng số của thuật toán.
        intent (str): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        category_mode (str): Ràng buộc phân loại dùng để lọc hoặc xếp hạng.
        use_vlm_rerank (bool): Cờ bật hoặc tắt hành vi tương ứng.
        vlm_top_k (int): Giá trị đầu vào phục vụ bước xử lý này.
        show_images (bool): Cờ bật hoặc tắt hành vi tương ứng.

    Returns:
        dict: Kết quả đã xử lý của hàm.
    """
    image_docs = debug_image_query(image_path, show_images=show_images)
    resolved_intent = infer_retrieval_goal(text_query) if intent == "auto" else intent
    category_info = infer_category_from_docs(image_docs)
    locked_category = category_info.get("category") if category_info.get("confident") else None

    if category_mode == "auto":
        resolved_category_mode = "hard_lock" if resolved_intent == "search" else "soft_boost"
    else:
        resolved_category_mode = category_mode

    if resolved_category_mode == "hard_lock" and locked_category is None:
        query_filter = None
        print("\n[WARN] Muốn hard_lock category nhưng image-only chưa đủ chắc, tạm không filter cứng.")
    elif resolved_category_mode == "hard_lock":
        query_filter = build_category_filter(locked_category)
    else:
        query_filter = None

    filter_text = f"category={locked_category}" if query_filter is not None else "không dùng filter cứng"
    print("\n=== Multimodal decision ===")
    print(f"intent        : {resolved_intent}")
    print(f"category_mode : {resolved_category_mode}")
    print(f"category_info : {category_info}")
    print(f"qdrant_filter : {filter_text}")

    print(f"\nTEXT QUERY: {text_query}")
    text_docs = search_products_by_text(text_query)
    print_docs_table("Text search results", text_docs, score_key="text_score")
    if show_images:
        show_retrieval_gallery("Top text retrieval results", text_docs, score_key="text_score")

    print(
        f"\nCOMPOSED QUERY: image_weight={image_weight} | "
        f"text_weight={text_weight} | threshold={composed_threshold}"
    )
    composed_docs = search_products_by_composed_image_text(
        image_path=image_path,
        text_query=text_query,
        image_weight=image_weight,
        text_weight=text_weight,
        top_k=MULTIMODAL_CANDIDATE_K,
        max_products=MULTIMODAL_CANDIDATE_K,
        score_threshold=composed_threshold,
        query_filter=query_filter,
    )
    print_docs_table("Composed image+text results", composed_docs, score_key="composed_score")
    if show_images:
        show_retrieval_gallery("Top composed image+text results", composed_docs, score_key="composed_score")

    metadata_docs = metadata_rerank_documents(
        docs=composed_docs,
        text_query=text_query,
        locked_category=locked_category,
        category_mode=resolved_category_mode,
        intent=resolved_intent,
        final_k=MULTIMODAL_FINAL_K,
    )
    print_docs_table("Metadata-aware rerank results", metadata_docs, score_key="metadata_rerank_score")
    if show_images:
        show_retrieval_gallery("Top metadata-aware rerank results", metadata_docs, score_key="metadata_rerank_score")

    final_docs = metadata_docs
    if use_vlm_rerank:
        print(f"\nVLM RERANK: model={QWEN_VL_MODEL} | top_k={vlm_top_k} | base_url={VLM_OLLAMA_BASE_URL}")
        final_docs = vlm_rerank_documents(
            query_image_path=image_path,
            text_query=text_query,
            docs=metadata_docs,
            intent=resolved_intent,
            locked_category=locked_category,
            top_k=vlm_top_k,
        )
        print_docs_table("VLM rerank results", final_docs[:MULTIMODAL_FINAL_K], score_key="vlm_score")
        if show_images:
            show_retrieval_gallery("Top VLM rerank results", final_docs[:MULTIMODAL_FINAL_K], score_key="vlm_score")

    image_text_summary = compare_retrieval_sets(image_docs, text_docs)
    image_composed_summary = compare_retrieval_sets(image_docs, composed_docs)
    text_composed_summary = compare_retrieval_sets(text_docs, composed_docs)
    clarification = build_clarification_diagnostic(
        text_query=text_query,
        intent=resolved_intent,
        category_mode=resolved_category_mode,
        category_info=category_info,
        composed_docs=composed_docs,
        final_docs=final_docs,
    )
    summary = {
        "intent": resolved_intent,
        "category_mode": resolved_category_mode,
        "category_info": category_info,
        "clarification": clarification,
        "image_vs_text": image_text_summary,
        "image_vs_composed": image_composed_summary,
        "text_vs_composed": text_composed_summary,
    }
    return {
        "image_docs": image_docs,
        "text_docs": text_docs,
        "composed_docs": composed_docs,
        "metadata_docs": metadata_docs,
        "final_docs": final_docs,
        "summary": summary,
    }


def debug_multimodal_retrieval(*args, **kwargs) -> dict:
    """Alias dễ nhớ hơn cho `debug_image_and_text`."""
    return debug_image_and_text(*args, **kwargs)


### BƯỚC 9E: Calibrate Threshold Và Ví Dụ Chạy

- **Tác dụng chính:** Thực hiện bước kiểm tra retrieval ảnh và truy vấn ảnh kết hợp văn bản.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `calibrate_image_threshold` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [52]:
def calibrate_image_threshold(
    sample_size: int = 20,
    top_k: int = 12,
    seed: int = 42,
) -> list[dict]:
    """Self-search ảnh trong dataset để xem positive match thường có score bao nhiêu.

    Args:
        sample_size (int): Giới hạn số phần tử được xử lý hoặc trả về.
        top_k (int): Giới hạn số phần tử được xử lý hoặc trả về.
        seed (int): Seed giúp kết quả lấy mẫu có thể lặp lại.

    Returns:
        list[dict]: Kết quả đã xử lý của hàm.
    """
    rng = random.Random(seed)
    reservoir = []
    seen = 0
    for item, rel_path, img_path in iter_products_with_main_image():
        seen += 1
        if len(reservoir) < sample_size:
            reservoir.append((item, rel_path, img_path))
        else:
            j = rng.randrange(seen)
            if j < sample_size:
                reservoir[j] = (item, rel_path, img_path)

    rows = []
    for idx, (item, rel_path, img_path) in enumerate(reservoir, start=1):
        product_id = str(item.get("product_id", ""))
        docs = search_products_by_image(
            image_path=img_path,
            top_k=top_k,
            max_products=top_k,
            score_threshold=None,
        )
        hit_rank = None
        hit_score = None
        top_score = None
        if docs:
            top_score = docs[0].metadata.get("image_search_score")
        for rank, doc in enumerate(docs, start=1):
            if str(doc.metadata.get("product_id", "")) == product_id:
                hit_rank = rank
                hit_score = doc.metadata.get("image_search_score")
                break
        row = {
            "query_product_id": product_id,
            "category": item.get("category", ""),
            "hit_rank": hit_rank,
            "hit_score": hit_score,
            "top_score": top_score,
            "image_path": str(img_path),
        }
        rows.append(row)
        print(
            f"[{idx}/{len(reservoir)}] {product_id} | "
            f"hit_rank={hit_rank} | hit_score={hit_score} | top_score={top_score}"
        )

    hit_scores = [float(row["hit_score"]) for row in rows if row["hit_score"] is not None]
    recall_at_k = sum(row["hit_rank"] is not None for row in rows) / max(1, len(rows))
    print("\n=== Threshold calibration summary ===")
    print(f"sample_size : {len(rows)}")
    print(f"recall@{top_k}   : {recall_at_k:.2%}")
    if hit_scores:
        print(f"min positive score : {min(hit_scores):.4f}")
        # Tóm tắt phân phối score để hiệu chỉnh ngưỡng retrieval.
        print(f"median positive    : {float(np.median(hit_scores)):.4f}")
        # Tóm tắt phân phối score để hiệu chỉnh ngưỡng retrieval.
        print(f"p10 positive       : {float(np.percentile(hit_scores, 10)):.4f}")
        # Tóm tắt phân phối score để hiệu chỉnh ngưỡng retrieval.
        print(f"p25 positive       : {float(np.percentile(hit_scores, 25)):.4f}")
        print(f"current threshold  : {IMAGE_SEARCH_SCORE_THRESHOLD}")
    return rows

# Ví dụ 1: random một ảnh trong dataset rồi image search
# sample = preview_random_main_image()
# image_docs = debug_image_query(sample["img_path"])

# Ví dụ 2: nhập ảnh bất kỳ trên máy
# image_docs = debug_image_query(r"D:\path\to\your_image.jpg")

# Ví dụ 3: true composed image+text retrieval
# result = debug_multimodal_retrieval(
#     image_path=r"D:\path\to\your_image.jpg",
#     text_query="giống ảnh này nhưng màu trắng, mặc đi làm",
#     image_weight=0.6,
#     text_weight=0.4,
#     intent="search",
#     category_mode="auto",
#     use_vlm_rerank=True,
#     vlm_top_k=3,
# )

# Ví dụ 4: calibrate threshold 0.15 bằng self-search, chạy ít mẫu trước vì sẽ load model ảnh
# threshold_rows = calibrate_image_threshold(sample_size=20, top_k=12)
